# Fine-tune XLM-R for text classification
Optional GPU workflow. Install `transformers`, `datasets`, `accelerate`, and compatible PyTorch separately. Requires labeled train/validation CSVs; choose your task label column. Keep a separate test split for final reporting.

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
BASE_MODEL = 'xlm-roberta-base'
LABEL_COLUMN = 'sentiment'
train_frame = pd.read_csv('../data/train/dataset.csv').dropna(subset=['text', LABEL_COLUMN])
valid_frame = pd.read_csv('../data/validation/dataset.csv').dropna(subset=['text', LABEL_COLUMN])
labels = sorted(train_frame[LABEL_COLUMN].astype(str).unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
def make_dataset(frame):
    data = Dataset.from_pandas(frame[['text', LABEL_COLUMN]].astype({LABEL_COLUMN: str}), preserve_index=False)
    def encode(batch):
        encoded = tokenizer(batch['text'], truncation=True, max_length=256)
        encoded['labels'] = [label2id.get(value, -1) for value in batch[LABEL_COLUMN]]
        return encoded
    return data.map(encode, batched=True).filter(lambda row: row['labels'] >= 0)
train_ds, valid_ds = make_dataset(train_frame), make_dataset(valid_frame)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(labels), label2id=label2id, id2label=id2label)
def metrics(output):
    logits, gold = output
    pred = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(gold, pred), 'f1_macro': f1_score(gold, pred, average='macro', zero_division=0)}

In [ ]:
output_dir = '../models/transformers/xlm-roberta-sentiment'
args = TrainingArguments(output_dir=output_dir, learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=16, num_train_epochs=3, weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1_macro', report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds, tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)